# Simulation design check (Part 2)

Pre-simulation checks for the interference simulation. Nothing here runs the simulation. It validates the inputs that the Part 2 design will lock, the same way `00_eda_pre_registration.ipynb` did for Part 1.

Inputs already decided (docs/Log.md, 2026-10-26)
- Base purchase probability per period, the Part 1 No E-Mail conversion rate, 0.5726%
- Demand lift from the discount, the Part 1 Mens vs No E-Mail lift, +0.6805pp

Sections 1 and 2 check how much regions should differ in their base rate. Sections 3 to 7 convert the rates to daily and try sizing the market from an 80% power target for the cluster design, which turned out too small for rationing. Section 8 checks the Hillstrom-sized market. Section 9 sizes the market from the mechanism, which is the locked rule. Section 10 sizes the sensitivity cases. Section 11 checks the supply ends away from the average region using the engine's own per-region rule. Section 12 works out the expected power of the cluster and switchback designs from a formula, without running the simulation.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
from scipy import stats
from load import load_hillstrom

df = load_hillstrom()
TABLES = Path.cwd().parent / "results" / "tables"
control = df[df["segment"] == "No E-Mail"].copy()
p_base = control["conversion"].mean()
print(f"rows: {len(df)}, No E-Mail rows: {len(control)}")
print(f"No E-Mail conversion rate: {p_base:.6f}")

rows: 64000, No E-Mail rows: 21306
No E-Mail conversion rate: 0.005726


## 1. Between-group spread in No E-Mail conversion

The simulated regions need a realistic spread in base rate. The closest thing in this project's data is how much No E-Mail conversion differs across customer groups. Two groupings are checked.
- A1, zip_code, 3 groups
- A2, zip_code x channel, 9 groups

Only the No E-Mail arm is used, since the base rate is the no-treatment rate.

The observed spread in group rates mixes two things, real differences between groups and pure sampling noise (a small group's rate bounces around even if its true rate equals everyone else's). The real part is estimated by subtracting the noise (method of moments).

- Observed variance of group rates, s^2 = sample variance of p_g across groups (ddof = 1)
- Average sampling variance, v = mean over groups of p_g (1 - p_g) / n_g
- Real between-group variance, tau^2 = max(0, s^2 - v)
- Real between-group SD, tau, and relative to the base rate, tau / p_base

A chi-square test of homogeneity on the group by converted table is shown beside it. It asks whether the groups differ at all beyond noise.

Grouping choice rule, set before looking. Prefer the estimate that is more reliable (more groups, so more degrees of freedom), not the one that gives a larger or smaller spread.

In [2]:
def spread(groups):
    g = control.groupby(groups)["conversion"].agg(n="size", conversions="sum", rate="mean").reset_index()
    g["sampling_var"] = g["rate"] * (1 - g["rate"]) / g["n"]
    s2 = g["rate"].var(ddof=1)
    v = g["sampling_var"].mean()
    tau2 = max(0.0, s2 - v)
    table = np.column_stack([g["conversions"], g["n"] - g["conversions"]])
    chi2, p_chi2, dof, _ = stats.chi2_contingency(table, correction=False)
    summary = {
        "grouping": " x ".join(groups),
        "n_groups": len(g),
        "df_spread": len(g) - 1,
        "smallest_group_n": int(g["n"].min()),
        "smallest_group_conversions": int(g["conversions"].min()),
        "observed_var_s2": s2,
        "mean_sampling_var_v": v,
        "tau2": tau2,
        "tau": np.sqrt(tau2),
        "tau_over_p_base": np.sqrt(tau2) / p_base,
        "chi2": chi2,
        "chi2_dof": dof,
        "chi2_p": p_chi2,
    }
    return g, summary

In [3]:
g_a1, s_a1 = spread(["zip_code"])
g_a1

,zip_code,n,conversions,rate,sampling_var
0,Rural,3139,26,0.008283,2.616848e-06
1,Surburban,9625,48,0.004987,5.155473e-07
2,Urban,8542,48,0.005619,6.541462e-07


In [4]:
g_a2, s_a2 = spread(["zip_code", "channel"])
g_a2

,zip_code,channel,n,conversions,rate,sampling_var
0,Rural,Multichannel,389,4,0.010283,0.000026
1,Rural,Phone,1404,10,0.007123,0.000005
2,Rural,Web,1346,12,0.008915,0.000007
3,Surburban,Multichannel,1171,7,0.005978,0.000005
4,Surburban,Phone,4157,20,0.004811,0.000001
5,Surburban,Web,4297,21,0.004887,0.000001
6,Urban,Multichannel,1046,7,0.006692,0.000006
7,Urban,Phone,3766,20,0.005311,0.000001
8,Urban,Web,3730,21,0.005630,0.000002


## 2. Side by side

Intermediate pieces first (s^2, v, tau^2), then tau and tau relative to the base rate.

In [5]:
summary = pd.DataFrame([s_a1, s_a2]).set_index("grouping")
pd.set_option("display.float_format", lambda x: f"{x:.6g}")
summary.T

grouping,zip_code,zip_code x channel
n_groups,3,9
df_spread,2,8
smallest_group_n,3139,389
smallest_group_conversions,26,4
observed_var_s2,3.05956e-06,3.54042e-06
mean_sampling_var_v,1.26218e-06,6.04222e-06
tau2,1.79738e-06,0
tau,0.00134066,0
tau_over_p_base,0.234133,0
chi2,4.54488,5.75132


## Save results

In [6]:
summary.reset_index().to_csv(TABLES / "sim_region_spread.csv", index=False)
pd.concat([g_a1.assign(grouping="zip_code"), g_a2.assign(grouping="zip_code x channel")]).to_csv(
    TABLES / "sim_region_spread_groups.csv", index=False)
print("saved sim_region_spread.csv, sim_region_spread_groups.csv")

saved sim_region_spread.csv, sim_region_spread_groups.csv


## 3. Daily rates

The test runs two weeks with daily periods (14 days). The base rate and lift from Part 1 are two-week values, so they are converted to daily rates.

- Two-week no-discount rate, p0 = No E-Mail conversion
- Lift, the Mens vs No E-Mail difference, computed here from the data
- Two-week discounted rate, p1 = p0 + lift
- Daily rate, p_day = 1 - (1 - p)^(1/14)

This assumes the chance of buying is the same every day of the window (Hillstrom has no purchase dates, so this is an assumption, not a validated fact). Check, 1 - (1 - p_day)^14 should give back p.

In [7]:
DAYS = 14
p_mens = df.loc[df["segment"] == "Mens E-Mail", "conversion"].mean()
p0 = p_base
lift = p_mens - p0
p1 = p0 + lift

def to_daily(p):
    return 1 - (1 - p) ** (1 / DAYS)

p0_day, p1_day = to_daily(p0), to_daily(p1)
print(f"two-week p0 {p0:.6f}, lift {lift:.6f}, p1 {p1:.6f}")
print(f"daily p0 {p0_day:.8f}, p1 {p1_day:.8f}, daily lift {p1_day - p0_day:.8f}")
print(f"check, back to two weeks: p0 {1 - (1 - p0_day) ** DAYS:.6f}, p1 {1 - (1 - p1_day) ** DAYS:.6f}")

two-week p0 0.005726, lift 0.006805, p1 0.012531
daily p0 0.00041010, p1 0.00090033, daily lift 0.00049023
check, back to two weeks: p0 0.005726, p1 0.012531


## 4. Region spread on the daily scale

Each region's two-week base rate is drawn from a beta distribution with mean p0 and SD = 23.41% of p0 (the A1 value from section 1, the larger of the two spread levels, used for sizing because it is the harder case for the cluster design).

Beta parameters from a mean m and SD s, k = m (1 - m) / s^2 - 1, a = m k, b = (1 - m) k.

The treated rate in a region is its own base rate plus the lift. Both are then converted to daily. The mean and variance of the daily rates across regions are found by numerical integration over the beta density, with a delta-method cross-check, Var(daily) is about (d p_day / d p)^2 Var(p), where d p_day / d p = (1/14)(1 - p)^(-13/14).

In [8]:
from scipy import integrate

tau_rel = s_a1["tau_over_p_base"]
m, s = p0, tau_rel * p0
k = m * (1 - m) / s**2 - 1
a_beta, b_beta = m * k, (1 - m) * k
print(f"two-week SD s {s:.6f}, k {k:.4f}, a {a_beta:.6f}, b {b_beta:.4f}")

def moments(f):
    pdf = stats.beta(a_beta, b_beta).pdf
    mean = integrate.quad(lambda p: f(p) * pdf(p), 0, 1, limit=200, points=[m])[0]
    second = integrate.quad(lambda p: f(p) ** 2 * pdf(p), 0, 1, limit=200, points=[m])[0]
    return mean, second - mean**2

mean0_day, var0_day = moments(lambda p: to_daily(p))
mean1_day, var1_day = moments(lambda p: to_daily(min(p + lift, 1.0)))
deriv0 = (1 / DAYS) * (1 - p0) ** (-(DAYS - 1) / DAYS)
deriv1 = (1 / DAYS) * (1 - p1) ** (-(DAYS - 1) / DAYS)
print(f"control daily, mean {mean0_day:.8f}, between-region var {var0_day:.6e}, delta-method var {deriv0**2 * s**2:.6e}")
print(f"treated daily, mean {mean1_day:.8f}, between-region var {var1_day:.6e}, delta-method var {deriv1**2 * s**2:.6e}")
print(f"daily effect at loose supply (no rationing), {mean1_day - mean0_day:.8f}")

two-week SD s 0.001341, k 3166.5539, a 18.131962, b 3148.4219


control daily, mean 0.00041016, between-region var 9.274071e-09, delta-method var 9.268629e-09
treated daily, mean 0.00090039, between-region var 9.393147e-09, delta-method var 9.387601e-09
daily effect at loose supply (no rationing), 0.00049023


## 5. Buyers per region from the 80% power target (superseded by section 9)

Kept as the record of why the power target was dropped as the sizing rule. Section 6 and 8 show the market it gives is far too small for rationing.

Locked inputs, 50 regions (25 discounted, 25 not), 14 days, alpha 0.05 two-sided, 80% power, cluster design at loose supply (no rationing, so the true effect is the full lift) and the larger spread.

The cluster test is a Welch t-test on each region's conversion, averaged over its N buyers x 14 days. The variance of one region's conversion has two parts,

V = between-region variance + E[p_day (1 - p_day)] / (14 N)

- SE of the difference = sqrt(V0 / 25 + V1 / 25)
- Noncentrality, ncp = daily effect / SE
- Welch df with equal group sizes, (V0/25 + V1/25)^2 / ((V0/25)^2 / 24 + (V1/25)^2 / 24)
- Power = P(|T| > t_crit) for a noncentral t with that df and ncp. The lower tail is negligible here and scipy returns NaN for it at very large ncp, so NaN there is read as 0

N is the smallest whole number of buyers per region that reaches 80%.

In [9]:
from scipy.optimize import brentq

K, ALPHA, POWER = 50, 0.05, 0.80
half = K // 2
within0 = integrate.quad(lambda p: to_daily(p) * (1 - to_daily(p)) * stats.beta(a_beta, b_beta).pdf(p), 0, 1, limit=200, points=[m])[0]
within1 = integrate.quad(lambda p: to_daily(min(p + lift, 1.0)) * (1 - to_daily(min(p + lift, 1.0))) * stats.beta(a_beta, b_beta).pdf(p), 0, 1, limit=200, points=[m])[0]
effect_day = mean1_day - mean0_day

def cluster_power(n, k=K, between0=var0_day, between1=var1_day, return_parts=False):
    h = k // 2
    v0 = between0 + within0 / (DAYS * n)
    v1 = between1 + within1 / (DAYS * n)
    se = np.sqrt(v0 / h + v1 / h)
    dfw = (v0 / h + v1 / h) ** 2 / ((v0 / h) ** 2 / (h - 1) + (v1 / h) ** 2 / (h - 1))
    t_crit = stats.t.ppf(1 - ALPHA / 2, dfw)
    ncp = effect_day / se
    lower_tail = np.nan_to_num(stats.nct.cdf(-t_crit, dfw, ncp), nan=0.0)
    power = stats.nct.sf(t_crit, dfw, ncp) + lower_tail
    if return_parts:
        return dict(n=n, v0=v0, v1=v1, se=se, df=dfw, t_crit=t_crit, ncp=ncp, power=power)
    return power

n_exact = brentq(lambda n: cluster_power(n) - POWER, 1, 1e8)
N = int(np.ceil(n_exact))
parts = cluster_power(N, return_parts=True)
print(f"within-region term, control {within0:.8f}, treated {within1:.8f}")
print(f"N solving power = 0.80 exactly: {n_exact:.2f}, rounded up: {N}")
for key, val in parts.items():
    print(f"  {key}: {val:.6g}")
print(f"power at N - 1: {cluster_power(N - 1):.6f}")
print(f"power at spread 0 with the same N: {cluster_power(N, between0=0.0, between1=0.0):.6f}")

within-region term, control 0.00040998, treated 0.00089957
N solving power = 0.80 exactly: 131.33, rounded up: 132
  n: 132
  v0: 2.31125e-07
  v1: 4.96173e-07
  se: 0.000170564
  df: 42.3726
  t_crit: 2.01756
  ncp: 2.87419
  power: 0.801929
power at N - 1: 0.799026
power at spread 0 with the same N: 0.811881


## 6. Daily demand per region

With N buyers per region, the expected number who want to buy on a given day, and its SD (binomial), at the average region.

- No discount, mean N p0_day, SD sqrt(N p0_day (1 - p0_day))
- Everyone discounted, mean N p1_day, SD sqrt(N p1_day (1 - p1_day))
- In the user-level design each arm has N / 2 buyers, so each arm's demand is half of these

This is the number that decides whether the tightness ends (held from the bundle) can be set as planned.

In [10]:
demand = pd.DataFrame({
    "world": ["no discount", "everyone discounted"],
    "daily_rate": [mean0_day, mean1_day],
    "expected_daily_demand": [N * mean0_day, N * mean1_day],
    "sd_daily_demand": [np.sqrt(N * mean0_day * (1 - mean0_day)), np.sqrt(N * mean1_day * (1 - mean1_day))],
})
demand["mean_minus_3sd"] = demand["expected_daily_demand"] - 3 * demand["sd_daily_demand"]
demand["mean_plus_3sd"] = demand["expected_daily_demand"] + 3 * demand["sd_daily_demand"]
demand["ratio_to_no_discount_demand"] = demand["expected_daily_demand"] / demand.loc[0, "expected_daily_demand"]
demand

,world,daily_rate,expected_daily_demand,sd_daily_demand,mean_minus_3sd,mean_plus_3sd,ratio_to_no_discount_demand
0,no discount,0.000410158,0.0541408,0.232634,-0.643762,0.752043,1
1,everyone discounted,0.000900389,0.118851,0.344593,-0.914928,1.15263,2.19523


## 7. Check on the region count (option C from the split decision)

The region count is locked at 50. As a check, the buyers per region and total buyers needed for 80% are shown for other region counts. If total buyers keep falling as regions rise, more regions would be cheaper overall, and that is reported, not acted on.

In [11]:
rows = []
for k_alt in [20, 30, 40, 50, 80, 100, 150, 200]:
    n_alt = brentq(lambda n: cluster_power(n, k=k_alt) - POWER, 1, 1e9)
    rows.append({"regions": k_alt, "buyers_per_region": int(np.ceil(n_alt)), "total_buyers": k_alt * int(np.ceil(n_alt))})
region_check = pd.DataFrame(rows)
region_check

,regions,buyers_per_region,total_buyers
0,20,373,7460
1,30,231,6930
2,40,168,6720
3,50,132,6600
4,80,80,6400
5,100,64,6400
6,150,42,6300
7,200,32,6400


## Save sizing results

In [12]:
sizing = pd.DataFrame([{
    "days": DAYS, "p0_2wk": p0, "lift_2wk": lift, "p1_2wk": p1, "p0_day": p0_day, "p1_day": p1_day,
    "spread_rel": tau_rel, "beta_a": a_beta, "beta_b": b_beta,
    "mean0_day": mean0_day, "mean1_day": mean1_day, "between_var0_day": var0_day, "between_var1_day": var1_day,
    "within0": within0, "within1": within1, "effect_day": effect_day,
    "regions": K, "alpha": ALPHA, "power_target": POWER, "n_exact": n_exact, "buyers_per_region": N,
    **{f"at_N_{key}": val for key, val in parts.items() if key != "n"},
}])
sizing.to_csv(TABLES / "sim_sizing.csv", index=False)
demand.to_csv(TABLES / "sim_daily_demand.csv", index=False)
region_check.to_csv(TABLES / "sim_region_count_check.csv", index=False)
print("saved sim_sizing.csv, sim_daily_demand.csv, sim_region_count_check.csv")

saved sim_sizing.csv, sim_daily_demand.csv, sim_region_count_check.csv


## 8. If the market were Hillstrom's own customers

Section 6 showed the power-sized market (132 buyers per region) has far too little daily demand for rationing. This checks the one market size the data gives, the 64,000 Hillstrom customers spread evenly over the 50 regions.

- Buyers per region, 64,000 / 50
- Daily demand per region, mean N p_day and SD sqrt(N p_day (1 - p_day)), at the average region
- P(demand = 0) and P(demand >= 2) on a region-day, binomial. Supply comes in whole units, so rationing can only bite on days with at least 2 interested buyers (with supply 1) or more

Also computed here, the buyers per region at which the real between-region variance equals the within-region noise over 14 days (between var = within / (14 N), so N = within / (14 between var)). Below this, sampling noise dominates and the spread level barely matters.

In [13]:
N_hill = len(df) // K
rows = []
for world, rate in [("no discount", mean0_day), ("everyone discounted", mean1_day)]:
    dist = stats.binom(N_hill, rate)
    rows.append({
        "world": world, "buyers_per_region": N_hill, "daily_rate": rate,
        "expected_daily_demand": N_hill * rate, "sd_daily_demand": np.sqrt(N_hill * rate * (1 - rate)),
        "p_zero": dist.pmf(0), "p_two_or_more": dist.sf(1),
    })
hill_market = pd.DataFrame(rows)
n_spread_equal = within0 / (DAYS * var0_day)
print(f"buyers per region if the market is Hillstrom's 64,000: {N_hill}")
print(f"buyers per region where between-region var equals within-region noise (control): {n_spread_equal:.1f}")
hill_market

buyers per region if the market is Hillstrom's 64,000: 1280
buyers per region where between-region var equals within-region noise (control): 3157.7


,world,buyers_per_region,daily_rate,expected_daily_demand,sd_daily_demand,p_zero,p_two_or_more
0,no discount,1280,0.000410158,0.525002,0.724422,0.59149,0.0978484
1,everyone discounted,1280,0.000900389,1.1525,1.07306,0.315683,0.320165


In [14]:
hill_market.assign(n_spread_equal=n_spread_equal).to_csv(TABLES / "sim_hillstrom_market.csv", index=False)
print("saved sim_hillstrom_market.csv")

saved sim_hillstrom_market.csv


## 9. Buyers per region from the mechanism (replaces the 80% power sizing)

Rule locked 2026-10-26. Find the smallest N (buyers per region) where, at the average region,

- eps = 1 / (50 regions x 14 days x 10,000 runs), so the whole loose-end scenario expects under one rationed region-day
- Loose-end supply S_L, the smallest whole number with P(everyone-discounted demand > S_L) <= eps
- Tight-end supply S_T, the largest whole number with P(no-discount demand < S_T) <= eps, and S_T >= 1
- At least 7 distinct whole-unit supply levels from S_T to S_L, so S_L - S_T + 1 >= 7
- The 5 band levels (evenly spaced from expected no-discount to expected everyone-discounted demand, rounded to whole units) are 5 distinct values, as the tightness grid needs

Daily demand is binomial, D ~ Binomial(N, p_day). Exact binomial tails, no normal approximation.

The smallest N is found by checking every N upward from 1. A halving search is not safe here, because the 5-distinct-levels condition is not monotone in N (it can pass, then fail, then pass again as N grows), so a halving search can stop above the smallest N. Fixed 2026-11-20 after the design review; the main case is unchanged.

The result is then checked against `sim.supply_levels`, the function the simulation engine uses, so the notebook and the engine cannot drift apart.

Afterwards, a check of how the ends hold for regions away from the average under the 23.41% spread (1st and 99th percentile region base rates), with supply scaled to each region's own expected no-discount demand at the same ratio.

In [15]:
RUNS = 10_000
eps = 1 / (K * DAYS * RUNS)

def ends(n, q0=p0_day, q1=p1_day):
    s_loose = int(stats.binom.isf(eps, n, q1))
    while stats.binom.sf(s_loose, n, q1) > eps:
        s_loose += 1
    k = int(stats.binom.ppf(eps, n, q0))
    while k >= 0 and stats.binom.cdf(k, n, q0) > eps:
        k -= 1
    s_tight = k + 1
    return s_tight, s_loose

def band_levels(n, q0=p0_day, q1=p1_day):
    return np.round(np.linspace(n * q0, n * q1, 5)).astype(int)

def rule_ok(n, q0=p0_day, q1=p1_day):
    s_tight, s_loose = ends(n, q0, q1)
    return s_tight >= 1 and s_loose - s_tight + 1 >= 7 and len(set(band_levels(n, q0, q1))) == 5

def smallest_n(q0, q1, start=1):
    n = start
    while not rule_ok(n, q0, q1):
        n += 1
    return n

N_mech = smallest_n(p0_day, p1_day)
s_tight, s_loose = ends(N_mech)
print(f"eps {eps:.6e}")
print(f"N from the mechanism rule: {N_mech} (N - 1 passes: {rule_ok(N_mech - 1)})")
print(f"expected daily demand, no discount {N_mech * p0_day:.4f}, everyone discounted {N_mech * p1_day:.4f}")
print(f"tight-end supply S_T {s_tight}, P(no-discount demand < S_T) {stats.binom.cdf(s_tight - 1, N_mech, p0_day):.3e}")
print(f"loose-end supply S_L {s_loose}, P(discounted demand > S_L) {stats.binom.sf(s_loose, N_mech, p1_day):.3e}")
print(f"supply levels from S_T to S_L: {s_loose - s_tight + 1}, band levels {band_levels(N_mech).tolist()}")
print(f"ends as ratios of expected no-discount demand: tight {s_tight / (N_mech * p0_day):.4f}, loose {s_loose / (N_mech * p0_day):.4f}")
print(f"total buyers over 50 regions: {K * N_mech:,}")

eps 1.428571e-07
N from the mechanism rule: 38426 (N - 1 passes: False)
expected daily demand, no discount 15.7584, everyone discounted 34.5960
tight-end supply S_T 1, P(no-discount demand < S_T) 1.428e-07
loose-end supply S_L 69, P(discounted demand > S_L) 8.064e-08
supply levels from S_T to S_L: 69, band levels [16, 20, 25, 30, 35]
ends as ratios of expected no-discount demand: tight 0.0635, loose 4.3786
total buyers over 50 regions: 1,921,300


In [16]:
import sim

engine_levels = sim.supply_levels(N_mech, p0_day, p1_day, eps)[0].tolist()
notebook_levels = [s_tight, *band_levels(N_mech).tolist(), s_loose]
print(f"engine levels   {engine_levels}")
print(f"notebook levels {notebook_levels}")
assert engine_levels == notebook_levels

engine levels   [1, 16, 20, 25, 30, 35, 69]
notebook levels [1, 16, 20, 25, 30, 35, 69]


In [17]:
ratio_tight = s_tight / (N_mech * p0_day)
ratio_loose = s_loose / (N_mech * p0_day)
rows = []
for pct in [0.01, 0.5, 0.99]:
    p0_r = stats.beta(a_beta, b_beta).ppf(pct)
    q0, q1 = to_daily(p0_r), to_daily(p0_r + lift)
    st_r = int(round(ratio_tight * N_mech * q0))
    sl_r = int(round(ratio_loose * N_mech * q0))
    rows.append({
        "region_percentile": pct, "two_week_rate": p0_r, "supply_tight": st_r, "supply_loose": sl_r,
        "p_tight_end_fails": stats.binom.cdf(st_r - 1, N_mech, q0),
        "p_loose_end_fails": stats.binom.sf(sl_r, N_mech, q1),
        "eps": eps,
    })
end_check = pd.DataFrame(rows)
end_check

,region_percentile,two_week_rate,supply_tight,supply_loose,p_tight_end_fails,p_loose_end_fails,eps
0,0.01,0.00307168,1,37,0.000215217,0.0293272,1.42857e-07
1,0.5,0.00562235,1,68,1.90179e-07,1.22938e-07,1.42857e-07
2,0.99,0.0092939,2,112,1.97344e-10,6.17971e-18,1.42857e-07


In [18]:
pd.DataFrame([{"eps": eps, "buyers_per_region": N_mech, "s_tight": s_tight, "s_loose": s_loose,
               "ratio_tight": ratio_tight, "ratio_loose": ratio_loose, "total_buyers": K * N_mech}]).to_csv(
    TABLES / "sim_market_size.csv", index=False)
end_check.to_csv(TABLES / "sim_end_check.csv", index=False)
print("saved sim_market_size.csv, sim_end_check.csv")

saved sim_market_size.csv, sim_end_check.csv


## 10. Inputs for the sensitivity checks

Two checks confirm the result does not depend on the Part 1 inputs, each changing one input and keeping the other.

- Higher base rate. The No E-Mail visit rate, a much higher real rate from the same experiment. Visiting is a different behavior from buying, so this only tests whether the pattern holds at a higher rate, it says nothing about visits. Lift stays +0.6805pp
- Smaller lift. The Womens vs No E-Mail lift. Base rate stays 0.5726%

Each case is converted to daily rates the same way as section 3 and resized with the section 9 rule (same eps, same 5 distinct band levels, same upward scan), since a different rate changes how many buyers the mechanism needs. Each case's levels are checked against `sim.supply_levels` too.

In [19]:
visit_base = control["visit"].mean()
p_womens = df.loc[df["segment"] == "Womens E-Mail", "conversion"].mean()
lift_womens = p_womens - p0
print(f"No E-Mail visit rate {visit_base:.6f} (n {len(control)}, visits {int(control['visit'].sum())})")
print(f"Womens conversion {p_womens:.6f}, Womens lift {lift_womens:.6f}")

def size_for(q0, q1):
    n = smallest_n(q0, q1)
    return n, ends(n, q0, q1)

rows = []
for case, b, l in [("main", p0, lift), ("higher base rate", visit_base, lift), ("smaller lift", p0, lift_womens)]:
    q0, q1 = to_daily(b), to_daily(b + l)
    n_case, (st, sl) = size_for(q0, q1)
    rows.append({"case": case, "base_2wk": b, "lift_2wk": l, "p0_day": q0, "p1_day": q1,
                 "buyers_per_region": n_case, "s_tight": st, "s_loose": sl, "band_levels": band_levels(n_case, q0, q1).tolist(),
                 "demand_no_discount": n_case * q0, "demand_discounted": n_case * q1})
    assert sim.supply_levels(n_case, q0, q1, eps)[0].tolist() == [st, *band_levels(n_case, q0, q1).tolist(), sl]
sensitivity_inputs = pd.DataFrame(rows)
sensitivity_inputs

No E-Mail visit rate 0.106167 (n 21306, visits 2262)
Womens conversion 0.008837, Womens lift 0.003111


,case,base_2wk,lift_2wk,p0_day,p1_day,buyers_per_region,s_tight,s_loose,band_levels,demand_no_discount,demand_discounted
0,main,0.00572609,0.00680501,0.000410098,0.000900328,38426,1,69,"[16, 20, 25, 30, 35]",15.7584,34.596
1,higher base rate,0.106167,0.00680501,0.00798485,0.00852623,5572,15,87,"[44, 45, 46, 47, 48]",44.4916,47.5082
2,smaller lift,0.00572609,0.00311106,0.000410098,0.000633829,38426,1,54,"[16, 18, 20, 22, 24]",15.7584,24.3555


In [20]:
sensitivity_inputs.to_csv(TABLES / "sim_sensitivity_inputs.csv", index=False)
print("saved sim_sensitivity_inputs.csv")

saved sim_sensitivity_inputs.csv


## 11. Supply ends away from the average region, engine rule

Section 9's end check scaled supply by one ratio for every region, a rule that was later replaced. The engine sets each region's ends from its own exact binomial demand (`sim.supply_levels`), with a floor of 1 unit at the tight end. This repeats the check with that rule, at the 1st, 50th and 99th percentile region base rates under the 23.41% spread, main case.

- Tight-end failure, P(no-discount demand < S_T), should be at most eps unless the 1 unit floor binds
- Loose-end failure, P(everyone-discounted demand > S_L), should be at most eps in every region

The section 9 table (`sim_end_check.csv`) is kept unchanged as the record of the old rule.

In [21]:
rows = []
for pct in [0.01, 0.5, 0.99]:
    p0_r = stats.beta(a_beta, b_beta).ppf(pct)
    q0, q1 = to_daily(p0_r), to_daily(p0_r + lift)
    lv = sim.supply_levels(N_mech, q0, q1, eps)[0]
    rows.append({
        "region_percentile": pct, "two_week_rate": p0_r, "levels": lv.tolist(),
        "p_tight_end_fails": stats.binom.cdf(lv[0] - 1, N_mech, q0),
        "p_loose_end_fails": stats.binom.sf(lv[-1], N_mech, q1),
        "band_distinct": len(set(lv[1:6].tolist())) == 5,
        "eps": eps,
    })
end_check_engine = pd.DataFrame(rows)
end_check_engine.to_csv(TABLES / "sim_end_check_engine.csv", index=False)
end_check_engine

,region_percentile,two_week_rate,levels,p_tight_end_fails,p_loose_end_fails,band_distinct,eps
0,0.01,0.00307168,"[1, 8, 13, 18, 23, 27, 58]",0.000215217,9.00249e-08,True,1.42857e-07
1,0.5,0.00562235,"[1, 15, 20, 25, 30, 34, 68]",1.90179e-07,1.22938e-07,True,1.42857e-07
2,0.99,0.0092939,"[4, 26, 30, 35, 40, 45, 83]",2.34362e-08,8.53899e-08,True,1.42857e-07


## 12. Expected power of the cluster and switchback designs, from a formula

Purpose. Check, before any simulation run, whether power can tell the cluster and switchback designs apart at each tightness level. Nothing here simulates. It uses exact binomial moments and the same t-test power formula as section 5.

Region conversion on a day is min(D, S), where D ~ Binomial(N, q) is demand and S the region's supply. For each region

- mean, E[min(D, S)] = sum over k < S of k P(D = k) + S P(D >= S)
- variance, E[min(D, S)^2] - E[min(D, S)]^2, with the second moment built the same way
- per buyer-day, divide the mean by N and the variance by N^2

Regions under spread are represented by 2,000 evenly spaced quantiles of the beta distribution, each with its own supply levels from `sim.supply_levels`. With no spread every region has the base rate.

Expected true effect, mean over regions of (discounted mean minus no-discount mean).

Cluster, Welch t on region rates, 25 regions per arm. A region rate averages 14 days, so its variance across regions is (variance of the region means) + (average within-region variance) / 14, for each arm. SE = sqrt(Var1 / 25 + Var0 / 25), Welch df as in section 5.

Switchback, one-sample t on 50 region differences, 49 df. A region's difference averages 7 discounted and 7 control days, so its variance is (variance of the region effects) + (average of (within var discounted + within var control) / 7). SE = sqrt(that / 50).

Power, P(|T| > t crit) with T noncentral t, noncentrality = expected true effect / SE. Near the tight end the effect is about zero and region outcomes are nearly constant, so the normal-theory formula is not reliable there, and power there is just a rejection rate.

In [22]:
G = 2_000
u = (np.arange(G) + 0.5) / G
LEVEL_NAMES = ["tight", "band 1", "band 2", "band 3", "band 4", "band 5", "loose"]


def min_moments(n, q, S):
    k = np.arange(S.max() + 1)
    pmf = stats.binom.pmf(k[None, :], n, q[:, None])
    below = k[None, :] < S[:, None]
    tail = stats.binom.sf(S - 1, n, q)
    m1 = (k * pmf * below).sum(axis=1) + S * tail
    m2 = (k**2 * pmf * below).sum(axis=1) + S**2 * tail
    return m1 / n, (m2 - m1**2) / n**2


def t_power(effect, se, df):
    t_crit = stats.t.ppf(1 - ALPHA / 2, df)
    ncp = effect / se
    lower = np.nan_to_num(stats.nct.cdf(-t_crit, df, ncp), nan=0.0)
    return t_crit, ncp, stats.nct.sf(t_crit, df, ncp) + lower


rows = []
for spread_name, p0_grid in [("0%", np.full(G, p0)), ("23.41%", stats.beta(a_beta, b_beta).ppf(u))]:
    q0g, q1g = to_daily(p0_grid), to_daily(np.minimum(p0_grid + lift, 1.0))
    levels = sim.supply_levels(N_mech, q0g, q1g, eps)
    for j, name in enumerate(LEVEL_NAMES):
        S = levels[:, j]
        mu0, w0 = min_moments(N_mech, q0g, S)
        mu1, w1 = min_moments(N_mech, q1g, S)
        effect = (mu1 - mu0).mean()

        var_c0 = mu0.var() + w0.mean() / DAYS
        var_c1 = mu1.var() + w1.mean() / DAYS
        h = K // 2
        se_c = np.sqrt(var_c0 / h + var_c1 / h)
        df_c = (var_c0 / h + var_c1 / h) ** 2 / ((var_c0 / h) ** 2 / (h - 1) + (var_c1 / h) ** 2 / (h - 1))
        tc_c, ncp_c, pow_c = t_power(effect, se_c, df_c)

        var_s = (mu1 - mu0).var() + ((w1 + w0) / (DAYS // 2)).mean()
        se_s = np.sqrt(var_s / K)
        tc_s, ncp_s, pow_s = t_power(effect, se_s, K - 1)

        rows.append({"spread": spread_name, "level": j, "level_name": name,
                     "avg_supply": S.mean(), "expected_true_effect": effect,
                     "cluster_se": se_c, "cluster_df": df_c, "cluster_ncp": ncp_c, "cluster_power": pow_c,
                     "switchback_se": se_s, "switchback_ncp": ncp_s, "switchback_power": pow_s})

expected_power = pd.DataFrame(rows)
expected_power.to_csv(TABLES / "sim_expected_power.csv", index=False)
expected_power

,spread,level,level_name,avg_supply,expected_true_effect,cluster_se,cluster_df,cluster_ncp,cluster_power,switchback_se,switchback_ncp,switchback_power
0,0%,0,tight,1,3.7169e-12,5.25707e-10,24,0.00707029,0.0500053,5.25707e-10,0.00707029,0.0500055
1,0%,1,band 1,16,4.4308e-05,3.18579e-06,24.0051,13.908,1,3.18579e-06,13.908,1
2,0%,2,band 2,20,0.000118665,4.73626e-06,24.0651,25.0545,1,4.73626e-06,25.0545,1
3,0%,3,band 3,25,0.000238624,5.49176e-06,25.124,43.4513,1,5.49176e-06,43.4513,1
4,0%,4,band 4,30,0.000352791,6.03736e-06,33.1384,58.4347,1,6.03736e-06,58.4347,1
5,0%,5,band 5,35,0.000434333,7.32394e-06,47.123,59.3031,1,7.32394e-06,59.3031,1
6,0%,6,loose,69,0.00049023,9.86728e-06,42.1129,49.6824,1,9.86728e-06,49.6824,1
7,23.41%,0,tight,1.321,2.98656e-10,5.61523e-06,48,5.31869e-05,0.05,4.72106e-09,0.0632604,0.0504409
8,23.41%,1,band 1,15.7595,4.07925e-05,2.67699e-05,47.9263,1.52382,0.320658,3.19372e-06,12.7727,1
9,23.41%,2,band 2,20.4705,0.000129267,2.73605e-05,47.9992,4.72459,0.996187,4.94165e-06,26.1588,1
